# 🧪 W4-D3 概念实验：高级 RAG —— 混合检索与重排序

> 配套阅读：`第4周-Day3-高级RAG混合检索与重排序.md`（方案矩阵与原理讲解在那边）
> 用一个 10 款糖水的小语料，**从零实现 BM25、语义向量检索、RRF 融合、重排序**，
> 再跑一个小型基准：量化每种方法各救回哪类查询
>
> 实验环境：纯 numpy + 标准库。语义向量用「手工语义标签维度」模拟真实
> embedding 模型学到的语义空间（透明、可复现、无需下载模型）。

## 实验 1：BM25 —— 关键词检索为什么至今没死

BM25 三要素：词频（饱和）、 IDF（稀有词更值钱）、文档长度归一。
中文用字符 bigram 分词。先看它在**字面匹配型**查询上的统治力。

In [ ]:
import numpy as np
from collections import Counter

# ---- 语料：10 款糖水（desc 刻意不含类别词，模拟真实场景的词汇鸿沟）----
docs = [
    ("草莓双皮奶", "新鲜草莓配嫩滑奶冻，酸甜可口"),
    ("杨枝甘露",   "芒果西柚西米露，港式经典，冰凉爽口"),
    ("冰镇绿豆沙", "陈皮绿豆沙，冰镇后清热去火，夏日必备"),
    ("姜撞奶",     "姜汁与水牛奶，温热滋补，冬日首选"),
    ("芝麻糊",     "黑芝麻细细研磨，香浓顺滑，传统糖水"),
    ("红豆沙",     "陈皮红豆沙，绵密香甜，广式经典"),
    ("芒果班戟",   "芒果奶油班戟，果肉饱满，人气之选"),
    ("龟苓膏",     "苦中回甘，清热降火，加蜜更佳"),
    ("黑芝麻汤圆", "黑芝麻馅汤圆，温热甜汤，冬至限定"),
    ("椰汁西米露", "椰香西米露，可加任意配料，清清爽爽"),
]
names = [d[0] for d in docs]
texts = [d[0] + d[1] for d in docs]

def bigrams(s):
    return [s[i:i+2] for i in range(len(s) - 1)]

# ---- BM25 ----
k1, b = 1.5, 0.75
N_doc = len(texts)
doc_tfs = [Counter(bigrams(t)) for t in texts]
df = Counter()
for tf in doc_tfs:
    df.update(set(tf))
avgdl = np.mean([sum(tf.values()) for tf in doc_tfs])
idf = {t: np.log(1 + (N_doc - c + 0.5) / (c + 0.5)) for t, c in df.items()}

def bm25(query, topk=5):
    qt = bigrams(query)
    scores = []
    for i, tf in enumerate(doc_tfs):
        dl = sum(tf.values())
        s = sum(idf.get(t, 0) * tf.get(t, 0) * (k1 + 1) /
                (tf.get(t, 0) + k1 * (1 - b + b * dl / avgdl)) for t in qt)
        scores.append(s)
    scores = np.array(scores)
    order = np.argsort(scores)[::-1][:topk]
    return [(names[i], scores[i]) for i in order]

print("查询: 「杨枝甘露多少钱」")
for name, s in bm25("杨枝甘露多少钱"):
    print(f"  {s:6.3f}  {name}")
print("\n→ 字面命中即高分：精确名词、型号、错误码类查询，BM25 至今是最强基线")

## 实验 2：语义检索 —— 词汇鸿沟的另一边

查询「天气热想解暑」：BM25 在语料里找不到"解暑/天热"这些字面（实验 1 的语料
刻意不含），**得分≈0、检索瘫痪**。给每款糖水挂语义标签（解暑/暖胃/招牌/甜品），
把标签拼进向量——模拟 embedding 模型把"想解暑"映射到"冰镇/清热"那个语义簇。

In [ ]:
TAGS = ["解暑", "暖胃", "招牌", "甜品"]
doc_tags = [
    ["招牌", "甜品"], ["解暑", "招牌"], ["解暑"], ["暖胃"], ["暖胃"],
    ["甜品"], ["招牌", "甜品"], ["解暑"], ["暖胃"], ["解暑", "甜品"],
]
vocab = sorted(idf.keys())
vidx = {t: i for i, t in enumerate(vocab)}

def build_vector(text, tags=None, tag_weight=1.6):
    # TF-IDF(bigram) ⊕ 语义标签维度 —— 模拟 embedding 的语义空间
    tf = Counter(bigrams(text))
    v = np.zeros(len(vocab) + len(TAGS))
    for t, f in tf.items():
        if t in vidx:
            v[vidx[t]] = f * idf[t]
    n0 = len(vocab)
    for tg in (tags or []):
        v[n0 + TAGS.index(tg)] = tag_weight
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else v

D_vec = np.array([build_vector(t, doc_tags[i]) for i, t in enumerate(texts)])

def vec_search(query, tag_hint=None, topk=5):
    qv = build_vector(query, [tag_hint] if tag_hint else None)
    sims = D_vec @ qv
    order = np.argsort(sims)[::-1][:topk]
    return [(names[i], sims[i]) for i in order]

q2 = "天气热想解暑"
print(f"查询: 「{q2}」")
print("  BM25 的尴尬:")
for name, s in bm25(q2, 3):
    print(f"    {s:6.3f}  {name}")
print("  语义检索（查询映射到「解暑」语义维度）:")
for name, s in vec_search(q2, tag_hint="解暑"):
    print(f"    {s:6.3f}  {name}")
print("\n→ 没有任何字面重叠，语义检索照样命中冰镇/清热款 —— 补上 BM25 的盲区")

## 实验 3：RRF 融合 —— 不比分数大小，只比排名

BM25 分数和余弦相似度**量纲不同、不可比**。Reciprocal Rank Fusion 只用排名：
`score(d) = Σ 1/(k + rank_d)`。跑一个 6 查询小基准，看混合检索是否稳定优于单路。

In [ ]:
def g(*idx):
    return {names[i] for i in idx}

queries = [
    ("草莓甜品有哪些",   g(0),          "甜品"),
    ("天气热想解暑",     g(1, 2, 7, 9), "解暑"),
    ("杨枝甘露多少钱",   g(1),          None),
    ("胃不舒服想暖暖的", g(3, 4, 8),    "暖胃"),
    ("招牌推荐",         g(0, 1, 6),    "招牌"),
    ("芒果做的东西",     g(1, 6),       None),
]

def bm25_r(q, tag=None):
    return [n for n, _ in bm25(q)]

def vec_r(q, tag=None):
    return [n for n, _ in vec_search(q, tag_hint=tag)]

def rrf_fuse(rank_lists, k=60, topk=5):
    agg = Counter()
    for rl in rank_lists:
        for r, name in enumerate(rl):
            agg[name] += 1 / (k + r + 1)
    return [n for n, _ in agg.most_common(topk)]

def evaluate(retriever):
    hit1 = hit3 = 0
    for q, gold, tag in queries:
        top = retriever(q, tag)
        hit1 += int(top[0] in gold)
        hit3 += int(len(set(top[:3]) & gold) > 0)
    return hit1 / len(queries), hit3 / len(queries)

def hybrid(q, tag=None):
    return rrf_fuse([bm25_r(q, tag), vec_r(q, tag)])

print(f"{'方法':<14}{'acc@1':>8}{'acc@3':>8}")
for name, fn in [("BM25", bm25_r), ("语义向量", vec_r), ("RRF 混合", hybrid)]:
    a1, a3 = evaluate(fn)
    print(f"{name:<14}{a1:>8.0%}{a3:>8.0%}")
print("\n→ 混合检索不做取舍：字面强的查询靠 BM25 一路，语义强的靠向量一路，RRF 稳吃两边")

## 实验 4：Cross-Encoder 重排序 —— 粗排 50 精排 5

Bi-encoder 只算"查询·文档"各自向量的相似度（可离线、可并行）；
Cross-Encoder 把**查询和文档拼在一起**过模型，能注意到细粒度交互——
但必须逐对计算，只能放在最后精排。用"逐字符细读打分"模拟 CE 的细粒度。

In [ ]:
def rerank_score(query, doc_name):
    """模拟 Cross-Encoder：细读查询与文档全文（含标签）的字符级交互"""
    idx = names.index(doc_name)
    full = doc_name + docs[idx][1] + "".join(doc_tags[idx])
    content = [c for c in query if c not in "的么有哪想不和"]
    hit = sum(1 for c in set(content) if c in full)
    return hit / max(len(set(content)), 1)

def hybrid_then_rerank(q, tag, recall_k=5, final_k=3):
    cands = rrf_fuse([bm25_r(q, tag), vec_r(q, tag)], topk=recall_k)
    scored = sorted(cands, key=lambda n: -rerank_score(q, n))
    return scored[:final_k]

# 单个例子：重排序如何搬动名次
q, gold, tag = queries[4]     # 「招牌推荐」
print(f"查询: 「{q}」（正确答案 = 招牌款）")
print("  粗排(RRF) top5:", rrf_fuse([bm25_r(q, tag), vec_r(q, tag)], topk=5))
print("  精排(CE)  top3:", hybrid_then_rerank(q, tag))
print("  CE 细读分数:", {n: round(rerank_score(q, n), 2) for n in
      rrf_fuse([bm25_r(q, tag), vec_r(q, tag)], topk=5)})

a1, a3 = evaluate(hybrid_then_rerank)
print(f"\n混合检索      acc@1: {evaluate(hybrid)[0]:.0%}  acc@3: {evaluate(hybrid)[1]:.0%}")
print(f"混合+CE 重排序 acc@1: {a1:.0%}  acc@3: {a3:.0%}")
print("\n→ 代价：CE 要对 50 个候选各跑一次前向（约 50× 单次开销），所以只精排 top 几十")

## 实验 5：四方法对决 —— 各自救回哪类查询

把 BM25 / 语义 / RRF 混合 / 混合+重排序 的 acc@1、acc@3 画在一起，
再打印每个查询的赢家——直观看到"没有银弹，只有组合拳"。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

methods = [("BM25", bm25_r), ("语义向量", vec_r), ("RRF混合", hybrid), ("混合+重排", hybrid_then_rerank)]
res = {name: evaluate(fn) for name, fn in methods}

fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.arange(len(methods)); w = 0.36
ax.bar(x - w/2, [res[m][0] for m, _ in methods], w, label="acc@1")
ax.bar(x + w/2, [res[m][1] for m, _ in methods], w, label="acc@3")
for i, m in enumerate(res):
    ax.text(i - w/2, res[m][0] + 0.01, f"{res[m][0]:.0%}", ha="center", fontsize=9)
    ax.text(i + w/2, res[m][1] + 0.01, f"{res[m][1]:.0%}", ha="center", fontsize=9)
ax.set_xticks(x, [m for m, _ in methods]); ax.set_ylim(0, 1.15)
ax.set_ylabel("准确率"); ax.set_title("糖水店小基准：四代检索方案对比（6 个查询）")
ax.legend(); plt.tight_layout(); plt.show()

for q, gold, tag in queries:
    wins = [m for m, fn in methods if fn(q, tag)[0] in gold]
    print(f"  「{q}」→ 赢家: {wins}")

## 结论

| 方法 | 强项 | 弱项 | 本实验表现 |
|---|---|---|---|
| BM25 | 精确字面（型号/名词/错误码） | 词汇鸿沟（解暑≠冰镇） | 字面查询 acc 高，语义查询瘫痪 |
| 语义向量 | 同义/改写/跨词表 | 专有名词易飘 | 语义查询全中，字面查询不稳 |
| RRF 混合 | 排名融合，免调权重 | 仍是粗排 | acc@3 满分（召回稳），acc@1 需精排兜底 |
| Cross-Encoder | 细粒度交互 | 逐对计算贵 | 精排 top5 → acc@1 再上一档 |

→ 深入阅读：同目录 `.md` 版本（查询改写 Multi-Query + 完整管道图）